<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/seq2one_return/stage_07_04_mlp_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_04 -  Modelo MLP (seq2one)**

El **MLP (Multi-Layer Perceptron)** es una red neuronal **feedforward** que introduce no linealidad en la relación entre la ventana histórica aplanada
(60 x 20 → 1200 features) y el target escalar.

A diferencia de los modelos lineales (Ridge/Lasso), el MLP puede capturar
interacciones no lineales entre las features, manteniendo una arquitectura
simple y controlable.

En este pipeline, el MLP se utiliza como **primer modelo no lineal**
de referencia, entrenado bajo el enfoque **seq2one**, y evaluado con las mismas
métricas que los modelos anteriores.


# **BLOQUE DE EJECUCIÓN COMPLETO**

# **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows_seq2one/seq2one_return/train_delta60_ws60.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows_seq2one/seq2one_return/test_delta60_ws60.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows_seq2one/seq2one_return/valid_delta60_ws60.npz"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows_seq2one/seq2one_return/train_delta90_ws60.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows_seq2one/seq2one_return/test_delta90_ws60.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows_seq2one/seq2one_return/valid_delta90_ws60.npz"))

#ESCALADOR GLOBAL
IN_SCALER = Path(os.environ.get("IN_SCALER", "data/scaled/scaler.joblib"))




In [3]:
#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
#IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
#IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [5]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z

IN_SCALER = DRIVE_DIR / IN_SCALER

#IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
#IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

# **2. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

# **3. Configuración**

In [7]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [8]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [9]:
# Construye un StageConfig leyendo ambos reports.
SUMMARY = """
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )
"""

In [10]:
#states_h60 = load_state_from_reports(horizon=60)
#states_h60

In [11]:
#states_h90 = load_state_from_reports(horizon=90)
#states_h90

# **4. Importar métricas comunes desde .py**

In [12]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [13]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


# **5. Carga de data windows**

In [14]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [15]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [16]:
# Cargar scaler
#scaler = joblib.load("/content/drive/MyDrive/neural_profit/data/scaled/scaler.joblib")

In [17]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [18]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


In [19]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (330144, 1200) (330144,)
H60 Valid: (70952, 1200) (70952,)
H60 Test : (70590, 1200) (70590,)
H90 Train: (330144, 1200) (330144,)
H90 Valid: (70952, 1200) (70952,)
H90 Test : (70590, 1200) (70590,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

# **6. Sanity Check**

In [20]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [21]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [22]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [23]:
summary = run_sanity_checks_all_horizons_seq2one(bundle_60, bundle_90)

#summary["h60"]["train"]

[sanity_check_seq2one] train_h60 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=1.008703
[sanity_check_seq2one] valid_h60 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=1.021581
[sanity_check_seq2one] test_h60 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=0.640061
OK h60 (h=60)
[sanity_check_seq2one] train_h90 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=1.008347
[sanity_check_seq2one] valid_h90 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=1.036845
[sanity_check_seq2one] test_h90 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=0.647445
OK h90 (h=90)


# **DEFINICIÓN DE MODELO**

# **7. Definición del modelo — placeholder**

## **7.1. Modelo MLP (Feedforward) — seq2one**

**Idea básica**

El **MLP (Multi-Layer Perceptron)** es una red neuronal **feedforward**
que aprende una relación **no lineal** entre la entrada aplanada
y el target escalar futuro.

A diferencia de los modelos lineales, el MLP puede capturar
**interacciones no lineales** entre las features, manteniendo una
arquitectura simple y controlable.

Formalmente, el modelo puede representarse como una composición de capas:

$$
\hat{y}_t = f_L\big( W_L \, f_{L-1}( \dots f_1(W_1 x_t + b_1) \dots ) + b_L \big)
$$

donde:
- $x_t \in \mathbb{R}^{1200}$ es la ventana histórica aplanada,
- $f_i(\cdot)$ son funciones de activación no lineales,
- $W_i, b_i$ son pesos y sesgos aprendidos.

---

**Regularización (MLP)**

**Riesgo:** Medio-alto, debido a la mayor capacidad del modelo.

La regularización **no es intrínseca** al MLP y debe definirse explícitamente:

- **Penalización L2 (weight decay):**
  - Controla la magnitud de los pesos.
- **Early stopping:**
  - Detiene el entrenamiento cuando el error en VALID deja de mejorar.
- **Arquitectura controlada:**
  - Pocas capas (1-2).
  - Número reducido de neuronas por capa.
- **Dropout (opcional):**
  - Solo si se observa sobreajuste claro.

La regularización es parte esencial del diseño del modelo y determina
su capacidad de generalización.

---

**Por qué el MLP es relevante en este proyecto**

- Entrada de **alta dimensión**: 60 x 20 = **1200 features**.
- Capacidad para modelar:
  - relaciones no lineales,
  - interacciones entre indicadores técnicos.
- Modelo:
  - más flexible que Ridge/Lasso,
  - más simple y estable que modelos temporales complejos.

El MLP actúa como el **primer baseline no lineal**, sirviendo de puente
entre modelos lineales y arquitecturas temporales más sofisticadas
(LSTM, TCN, Transformer).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Número de capas ocultas: 1–2
- Neuronas por capa: moderado (por ejemplo, 64–128)
- Función de activación: ReLU
- Regularización L2 (weight decay): fija
- Early stopping: activado
- Dropout: desactivado inicialmente
- **Sin validación interna automática** (la evaluación se realiza externamente en VALID)

El ajuste fino de la arquitectura y la regularización se aborda en etapas posteriores.


## **7.2. Imports (PyTorch) + semillas**

In [24]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [25]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

## **7.3. Dataset/DataLoader desde bundle**

In [26]:
def make_loaders_from_bundle(
    bundle,
    *,
    batch_size: int = 4096,
    num_workers: int = 0,
) -> dict:
    """
    Convierte bundle {train/valid/test} a DataLoaders PyTorch.
    Espera X: (n, 1200) y y: (n,)
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)  # (n,1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders



In [27]:
loaders_60 = make_loaders_from_bundle(bundle_60, batch_size=16384)
loaders_90 = make_loaders_from_bundle(bundle_90, batch_size=16384)

## **7.3. Definición del modelo MLP (simple y controlado)**

In [28]:
class MLPSeq2One(nn.Module):
    def __init__(self, in_dim: int = 1200, hidden_dim: int = 128, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)


## **7.4. Entrenamiento con early stopping (VALID)**

In [29]:
@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)


def train_mlp(
    loaders: dict,
    *,
    in_dim: int = 1200,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
):
    """
    Entrena un MLP seq2one usando TRAIN y early stopping en VALID (por MSE).
    """
    model = MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loaders["train"]:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

        valid_mse = evaluate_mse(model, loaders["valid"], device)

        print(f"epoch={epoch:02d} | valid_mse={valid_mse:.6f}")

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}). Best valid_mse={best_valid:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


## **7.5. Entrenar MLP para H60 y H90**


In [30]:
# Chequeo rápido de que el modelo y los datos efectivamente van a GPU
xb, yb = next(iter(loaders_60["train"]))
xb = xb.to(device)
yb = yb.to(device)

test_model = MLPSeq2One(in_dim=1200, hidden_dim=128, dropout=0.0).to(device)
out = test_model(xb)

print("device:", device)
print("xb.device:", xb.device)
print("model.device:", next(test_model.parameters()).device)
print("out.device:", out.device)

device: cuda
xb.device: cuda:0
model.device: cuda:0
out.device: cuda:0


In [31]:
mlp_60 = train_mlp(
    loaders_60,
    in_dim=1200,
    hidden_dim=128,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    device=device,
)



epoch=01 | valid_mse=1.111100
epoch=02 | valid_mse=1.079389
epoch=03 | valid_mse=1.094609
epoch=04 | valid_mse=1.071504
epoch=05 | valid_mse=1.073561
epoch=06 | valid_mse=1.084503
epoch=07 | valid_mse=1.097925
epoch=08 | valid_mse=1.077070
epoch=09 | valid_mse=1.107992
Early stopping (patience=5). Best valid_mse=1.071504


In [32]:
mlp_90 = train_mlp(
    loaders_90,
    in_dim=1200,
    hidden_dim=128,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    device=device,
)

epoch=01 | valid_mse=1.093259
epoch=02 | valid_mse=1.100636
epoch=03 | valid_mse=1.116408
epoch=04 | valid_mse=1.108480
epoch=05 | valid_mse=1.107643
epoch=06 | valid_mse=1.109572
Early stopping (patience=5). Best valid_mse=1.093259


## **7.6. Predicciones MLP**


In [33]:
import numpy as np
import torch

@torch.no_grad()
def predict_mlp(model, X: np.ndarray, *, device: torch.device, batch_size: int = 32768) -> np.ndarray:
    """
    Predice con un modelo MLP PyTorch en batches.
    Retorna shape (n_samples,)
    """
    model.eval()

    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        yb = model(xb).squeeze(-1)          # (batch,)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)


In [34]:
# VALID (H=60)
X_valid_60 = bundle_60["valid"]["X"]
y_pred_valid_60 = predict_mlp(mlp_60, X_valid_60, device=device)

# TEST (H=60)
X_test_60 = bundle_60["test"]["X"]
y_pred_test_60 = predict_mlp(mlp_60, X_test_60, device=device)

In [35]:
# VALID (H=90)
X_valid_90 = bundle_90["valid"]["X"]
y_pred_valid_90 = predict_mlp(mlp_90, X_valid_90, device=device)

# TEST (H=90)
X_test_90 = bundle_90["test"]["X"]
y_pred_test_90 = predict_mlp(mlp_90, X_test_90, device=device)

# **8. Métricas ML**

In [36]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])


In [37]:
# ============================================================
# H=60
# ============================================================
y_valid_60 = bundle_60["valid"]["y"]
y_test_60  = bundle_60["test"]["y"]

metrics_valid_60 = compute_seq2one_metrics(y_valid_60, y_pred_valid_60, compute_r2=True)
metrics_test_60  = compute_seq2one_metrics(y_test_60,  y_pred_test_60,  compute_r2=True)

df_valid_60 = metrics_to_df(metrics_valid_60, model="mlp", split="valid", horizon=60)
df_test_60  = metrics_to_df(metrics_test_60,  model="mlp", split="test",  horizon=60)

# ============================================================
# H=90
# ============================================================
y_valid_90 = bundle_90["valid"]["y"]
y_test_90  = bundle_90["test"]["y"]

metrics_valid_90 = compute_seq2one_metrics(y_valid_90, y_pred_valid_90, compute_r2=True)
metrics_test_90  = compute_seq2one_metrics(y_test_90,  y_pred_test_90,  compute_r2=True)

df_valid_90 = metrics_to_df(metrics_valid_90, model="mlp", split="valid", horizon=90)
df_test_90  = metrics_to_df(metrics_test_90,  model="mlp", split="test",  horizon=90)

In [38]:
# ============================================================
# Tabla final (VALID+TEST, H60+H90)
# ============================================================
df_mlp_metrics = pd.concat([df_valid_60, df_valid_90, df_test_60, df_test_90], ignore_index=True)
df_mlp_metrics = df_mlp_metrics.sort_values(["split", "horizon_min"]).reset_index(drop=True)

df_mlp_metrics

,model,split,horizon_min,MAE,RMSE,R2,DA
0,mlp,test,60,0.484073,0.651302,-0.035433,0.502522
1,mlp,test,90,0.480513,0.653726,-0.019497,0.503216
2,mlp,valid,60,0.665836,1.035135,-0.026710,0.499056
3,mlp,valid,90,0.667726,1.045590,-0.016940,0.493390


# **9. Guardar artefactos para Stage_08**

In [39]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_return",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"metrics_{name}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path


In [40]:
save_seq2one_metrics(
    df_mlp_metrics,
    name="mlp",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_return/metrics_mlp.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_return/metrics_mlp.parquet')